In [8]:
import pandas as pd

df = pd.read_csv('northwoodfresh_sales.csv')

print(df.shape)

(48, 3)


In [9]:
print(df.dtypes)
print(df.head())

region         str
month        int64
revenue    float64
dtype: object
      region  month    revenue
0  Northeast      1  532914.57
1  Northeast      2  516405.13
2  Northeast      3  536839.90
3  Northeast      4  559598.78
4  Northeast      5  513912.01


In [10]:
print(f"Row count: {len(df)}")

Row count: 48


The dataset contains 48 rows and 3 columns: region, month, and revenue. The region column includes four categories, including Northeast, Southeast, Midwest, and West. All data types loaded correctly, with revenue as a float and month as an integer.

In [11]:
summary = df.groupby('region')['revenue'].agg(['mean', 'median', 'std'])
print(summary)

                    mean      median           std
region                                            
Midwest    505069.950000  499340.025  22873.708944
Northeast  527694.838333  524659.850  19350.267268
Southeast  411927.701667  415597.885  55608.194364
West       580188.961667  583852.910  26790.433030


In [12]:
def iqr(series):
    return series.quantile(0.75) - series.quantile(0.25)

summary = df.groupby('region')['revenue'].agg(['mean', 'median', 'std', iqr])
print(summary)

                    mean      median           std         iqr
region                                                        
Midwest    505069.950000  499340.025  22873.708944  22767.0200
Northeast  527694.838333  524659.850  19350.267268  25196.4575
Southeast  411927.701667  415597.885  55608.194364  95727.5700
West       580188.961667  583852.910  26790.433030  31617.8500


Southeast's IQR (95,728) spreads out dramatically bigger than the other three regions (22,767–31,618) because of the mid-year step-down


In [13]:
import plotly.express as px

southeast = df[df['region'] == 'Southeast']

fig = px.histogram(southeast, x='revenue', nbins=12, title='Southeast Monthly Revenue Distribution')
fig.update_layout(xaxis_title='Revenue (USD)', yaxis_title='Count')
fig.show()

The histogram confirms the distribution of monthly revenue for the Southeast region across all 12 months is bimodal; two distinct clusters instead of one smooth curve, reflecting the pre-drop months (~480k) and post-drop months (~370k) rather than a single stable average.

In [14]:
fig2 = px.box(df, x='region', y='revenue', title='Monthly Revenue by Region')
fig2.update_layout(xaxis_title='Region', yaxis_title='Revenue (USD)')
fig2.show()

The box plot confirms the monthly revenue by region across all 12 months. Confirming Southeast has a dramatically wider spread than the other three regions, onsistent with its much larger IQR (95,728 vs. 22,767-31,618). Northeast, Midwest, and West show tight, consistent boxes clustered around their respective baselines, while Southeast's box and whiskers span nearly the full 340k-490k range, visual evidence of the mid-year step-down rather than simple random variation.

In [15]:
southeast = df[df['region'] == 'Southeast']
before = southeast[southeast['month'] <= 6]['revenue']
after = southeast[southeast['month'] >= 7]['revenue']

effect_dollars = before.mean() - after.mean()
effect_percent = (effect_dollars / before.mean()) * 100

print(f"Effect size: ${effect_dollars:,.2f}")
print(f"Effect size: {effect_percent:.2f}%")

Effect size: $98,887.74
Effect size: 21.43%


In [18]:
import numpy as np

std1 = before.std(ddof=1)
std2 = after.std(ddof=1)
n1 = len(before)
n2 = len(after)

SE = np.sqrt((std1**2 / n1) + (std2**2 / n2))
print(f"Standard Error: {SE:,.2f}")

ci_lower = effect_dollars - 1.96 * SE
ci_upper = effect_dollars + 1.96 * SE

print(f"95% CI for the difference: (${ci_lower:,.2f}, ${ci_upper:,.2f})")

Standard Error: 12,488.17
95% CI for the difference: ($74,410.93, $123,364.55)


In [19]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(before, after)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")

T-statistic: 7.9185
P-value: 0.000013


Southeast's monthly revenue dropped by $98,887.74 (21.43%) after the mid-year change. The 95% confidence interval for this difference is $74,410.93 to $123,364.55;  notably, this range doesn't include zero, meaning "no real difference" isn't a plausible explanation. The t-test confirms this: p = 0.000013, far below the typical 0.05 threshold, indicating the drop is highly unlikely to be due to random chance. Together, these three numbers show a real, substantial, and statistically significant decline in Southeast's revenue.